In [ ]:
from typing import Literal

from dotenv import  load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import MessagesState, StateGraph,START,END
from langchain.tools import tool
from loguru import logger
from langchain.messages import HumanMessage,ToolMessage
load_dotenv(override = True)

#1. ツール呼び出しを追加
@tool(parse_docstring=True)
def get_weather(city:str) ->str:
    """
    指定された都市の当日の天気を照会する

    Args:
        city:都市名
    """
    return f"{city} は晴れ、微風です"

@tool(parse_docstring=True)
def get_news(home_or_abroad:bool) ->str:
    """
    国内外のニュースを照会する

    Args:
        home_or_abroad: 国内または海外のニュースを照会する True:国内ニュース False:海外ニュース
    """
    if home_or_abroad:
        return "Kimi の新モデルがリリースされました"
    return "Anthropic が新モデルへのアクセスを一時停止しました"

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)
#2. ツールをモデルにバインド
tools_by_name = {
    "get_weather":get_weather,
    "get_news":get_news
}

tools = [get_weather,get_news]

model_with_tools = model.bind_tools(tools=tools)


#3. 状態を宣言 -> 渡すときは直接 HumanMessage を書き込む
class OverAllState(MessagesState):
    output:str

#4. ノードを定義
def llm_node(state:OverAllState) -> OverAllState:
    msg = state["messages"]
    # 大規模言語モデルを呼び出す
    res = model_with_tools.invoke(msg)

    return {
        "messages":[res],
        "output":res.content
    }

def tool_node(state:OverAllState) -> OverAllState:
    last_msg = state["messages"][-1]
    if not last_msg.tool_calls:
        return {}

    tool_msgs = []
    for tool_call in last_msg.tool_calls:
        #1. tool_call 内の name から対応するツール関数を取得
        tool = tools_by_name[tool_call["name"]]
        logger.info("ツール{}が呼び出されました。対応する tool_call:{}",tool_call["name"],tool_call)
        #2. ツールを実行
        tool_res = tool.invoke(tool_call["args"])
        tool_msg = ToolMessage(
            name = tool_call["name"],
            content= tool_res,
            tool_call_id = tool_call["id"]
        )
        tool_msgs.append(tool_msg)
    return {
        "messages":tool_msgs
    }

def router(state:OverAllState) -> Literal["tool_node",END]:
    if state["messages"][-1].tool_calls:
        return "tool_node"
    return END

#5. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_node",llm_node)
builder.add_node("tool_node",tool_node)

builder.add_edge(START,"llm_node")
builder.add_conditional_edges("llm_node",router,path_map=["tool_node",END])
builder.add_edge("tool_node","llm_node")

graph = builder.compile()

from IPython.display import display
display(graph)

res = graph.invoke({"messages":[HumanMessage("今日の東京の天気はどうですか？国内にはどんなニュースがありますか？")]})
print(res)
for msg in res["messages"]:
    msg.pretty_print()
